In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

**Import Libraries**

In [3]:
import os
import random
import shutil
import numpy as np

from PIL import Image

import tensorflow as tf

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D
)

from tensorflow.keras.applications import (
    ResNet50,
    DenseNet121
)

**Dataset Paths**

In [4]:
normal_path = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/train/NORMAL"

pneumonia_path = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/train/PNEUMONIA"

**Select 50 Images**

In [5]:
random.seed(42)

normal_images = random.sample(
    os.listdir(normal_path),
    25
)

pneumonia_images = random.sample(
    os.listdir(pneumonia_path),
    25
)

print("NORMAL:", len(normal_images))
print("PNEUMONIA:", len(pneumonia_images))

NORMAL: 25
PNEUMONIA: 25


**Create Folder Structure**

In [8]:
base_dir = "/kaggle/working/mini_dataset"

folders = [
    "train/NORMAL",
    "train/PNEUMONIA",
    "test/NORMAL",
    "test/PNEUMONIA"
]

for folder in folders:
    os.makedirs(
        os.path.join(base_dir, folder),
        exist_ok=True
    )

**Split Images**

In [10]:
normal_train, normal_test = train_test_split(
    normal_images,
    test_size=0.2,
    random_state=42
)

pneumonia_train, pneumonia_test = train_test_split(
    pneumonia_images,
    test_size=0.2,
    random_state=42
)

**Copy Images**

In [11]:
for img in normal_train:
    shutil.copy(
        os.path.join(normal_path, img),
        os.path.join(base_dir, "train/NORMAL", img)
    )

for img in normal_test:
    shutil.copy(
        os.path.join(normal_path, img),
        os.path.join(base_dir, "test/NORMAL", img)
    )

for img in pneumonia_train:
    shutil.copy(
        os.path.join(pneumonia_path, img),
        os.path.join(base_dir, "train/PNEUMONIA", img)
    )

for img in pneumonia_test:
    shutil.copy(
        os.path.join(pneumonia_path, img),
        os.path.join(base_dir, "test/PNEUMONIA", img)
    )

**Create Generators**

In [12]:
IMG_SIZE = (128,128)
BATCH_SIZE = 8

train_datagen = ImageDataGenerator(
    rescale=1./255
)

test_datagen = ImageDataGenerator(
    rescale=1./255
)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, "train"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    os.path.join(base_dir, "test"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 40 images belonging to 2 classes.
Found 10 images belonging to 2 classes.


**CNN Model**

In [13]:
cnn_model = Sequential([

    Conv2D(
        32,
        (3,3),
        activation='relu',
        input_shape=(128,128,3)
    ),

    MaxPooling2D(2,2),

    Conv2D(
        64,
        (3,3),
        activation='relu'
    ),

    MaxPooling2D(2,2),

    Flatten(),

    Dense(
        128,
        activation='relu'
    ),

    Dropout(0.5),

    Dense(
        1,
        activation='sigmoid'
    )
])

cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

cnn_model.fit(
    train_generator,
    epochs=5
)

cnn_loss, cnn_acc = cnn_model.evaluate(
    test_generator
)

print("CNN Accuracy:", cnn_acc)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-06-13 11:15:59.890621: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 167ms/step - accuracy: 0.5750 - loss: 1.1430
Epoch 2/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 155ms/step - accuracy: 0.5750 - loss: 0.7907
Epoch 3/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 154ms/step - accuracy: 0.7750 - loss: 0.5727
Epoch 4/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 155ms/step - accuracy: 0.9000 - loss: 0.4302
Epoch 5/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 160ms/step - accuracy: 0.9250 - loss: 0.2704
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 1.0000 - loss: 0.2616 
CNN Accuracy: 1.0


**ResNet50 Model**

In [14]:
resnet_base = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

resnet_base.trainable = False

resnet_model = Sequential([

    resnet_base,

    GlobalAveragePooling2D(),

    Dense(
        128,
        activation='relu'
    ),

    Dropout(0.5),

    Dense(
        1,
        activation='sigmoid'
    )
])

resnet_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

resnet_model.fit(
    train_generator,
    epochs=5
)

resnet_loss, resnet_acc = resnet_model.evaluate(
    test_generator
)

print("ResNet50 Accuracy:", resnet_acc)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 8s 244ms/step - accuracy: 0.4750 - loss: 0.7671
Epoch 2/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 253ms/step - accuracy: 0.5000 - loss: 0.7654
Epoch 3/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 237ms/step - accuracy: 0.5250 - loss: 0.7159
Epoch 4/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 247ms/step - accuracy: 0.5750 - loss: 0.6943
Epoch 5/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 249ms/step - accuracy: 0.4500 - loss: 0.7038
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step - accuracy: 0.5000 - loss: 0.6938
ResNet50 Accuracy: 0.5


**DenseNet121 Model**

In [15]:
densenet_base = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

densenet_base.trainable = False

densenet_model = Sequential([

    densenet_base,

    GlobalAveragePooling2D(),

    Dense(
        128,
        activation='relu'
    ),

    Dropout(0.5),

    Dense(
        1,
        activation='sigmoid'
    )
])

densenet_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

densenet_model.fit(
    train_generator,
    epochs=5
)

densenet_loss, densenet_acc = densenet_model.evaluate(
    test_generator
)

print("DenseNet121 Accuracy:", densenet_acc)

29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 14s 233ms/step - accuracy: 0.5000 - loss: 1.2620
Epoch 2/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 238ms/step - accuracy: 0.6250 - loss: 0.8799
Epoch 3/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 236ms/step - accuracy: 0.7250 - loss: 0.4955
Epoch 4/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 237ms/step - accuracy: 0.8250 - loss: 0.3045
Epoch 5/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 238ms/step - accuracy: 0.8500 - loss: 0.3439
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 91ms/step - accuracy: 1.0000 - loss: 0.1629
DenseNet121 Accuracy: 1.0


**Final Comparison**

In [16]:
print("\nMODEL COMPARISON")
print("---------------------------")

print("CNN         :", cnn_acc)
print("ResNet50    :", resnet_acc)
print("DenseNet121 :", densenet_acc)


MODEL COMPARISON
---------------------------
CNN         : 1.0
ResNet50    : 0.5
DenseNet121 : 1.0
